In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# !pip install openai -q

In [7]:
# Cell 1: Setup
import pandas as pd
import json
import time
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

BASE_DIR = '/content/drive/MyDrive/capstone/data'
df = pd.read_csv(f'{BASE_DIR}/mayo_clinic_diseases_raw.csv')
print(f"Loaded {len(df)} rows")

Loaded 1177 rows


In [8]:
# Cell 2: Cleaning function

PROGRESS_FILE = f'{BASE_DIR}/cleaning_progress.csv'

def clean_row(name, symptoms_text, risk_text):
    symptoms_text = str(symptoms_text)[:4000] if pd.notna(symptoms_text) else ''
    risk_text = str(risk_text)[:5000] if pd.notna(risk_text) else ''

    prompt = f"""You are a medical data cleaning assistant. Process this disease entry.

DISEASE: {name}

=== SYMPTOMS TEXT ===
{symptoms_text}

=== RISK FACTORS TEXT ===
{risk_text}

Return ONLY a valid JSON object with exactly 3 keys. No markdown, no backticks, no explanation.

1. "symptoms": Extract only symptoms actually mentioned in the text.
   - Use simple patient language, include body location (e.g. "chest pain" not "pain")
   - Exclude: the disease name, doctor names, drugs, procedures, risk factors, advice, citations, forum posts
   - Format: keywords separated by | on one line. Empty string if none.

2. "risk_factors": Extract risk factors that work as yes/no patient questions.
   Each must make sense in: "Do you have/experience [X]?"
   - Exclude: geographic locations, vague phrases, advice, treatments, the disease name, complications, citations
   - Max 12, merge similar items, simple language
   - Format: keywords separated by | on one line. Empty string if none.

3. "specialist": The single most appropriate medical specialist for this disease.
   Use standard medical specialist titles (e.g. cardiologist, dermatologist, neurologist,
   gastroenterologist, oncologist, pulmonologist, rheumatologist, endocrinologist,
   nephrologist, urologist, ophthalmologist, psychiatrist, orthopedic surgeon,
   neurosurgeon, vascular surgeon, hepatologist, geriatrician, neonatologist,
   perinatologist, andrologist, toxicologist, sports medicine specialist,
   pain management specialist, palliative care specialist, sleep medicine specialist,
   or any other real medical specialty that fits best).

Return ONLY: {{"symptoms": "...", "risk_factors": "...", "specialist": "..."}}"""

    for attempt in range(5):
        try:
            # Collect streamed response
            chunks = []
            completion = client.chat.completions.create(
                model="meta/llama-3.3-70b-instruct",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                top_p=0.7,
                max_tokens=1024,
                stream=True
            )
            for chunk in completion:
                if chunk.choices and chunk.choices[0].delta.content:
                    chunks.append(chunk.choices[0].delta.content)

            text = ''.join(chunks).strip()

            # Clean markdown fences
            if text.startswith('```'):
                text = text.split('\n', 1)[1] if '\n' in text else text[3:]
                if text.endswith('```'): text = text[:-3]
                text = text.strip()
            if text.startswith('json'): text = text[4:].strip()

            return json.loads(text)

        except json.JSONDecodeError:
            if attempt < 4: time.sleep(2)
        except Exception as e:
            wait = min(2 ** attempt * 5, 60)
            time.sleep(wait)

    return {"symptoms": "", "risk_factors": "", "specialist": ""}

# Quick test
test = clean_row(df['disease_name'].iloc[0], df['symptoms'].iloc[0], df['risk_factors'].iloc[0])
print(f"{df['disease_name'].iloc[0]}: {json.dumps(test, indent=2)}")

Atrial fibrillation: {
  "symptoms": "fast heartbeat | chest pain | dizziness | fatigue | lightheadedness | shortness of breath | weakness",
  "risk_factors": "high blood pressure | diabetes | heart conditions | family history | obesity | thyroid disease | kidney disease | lung disease | sleep apnea",
  "specialist": "cardiologist"
}


In [9]:
# Cell 3: Process all rows (resumable)

if os.path.exists(PROGRESS_FILE):
    done_df = pd.read_csv(PROGRESS_FILE)
    progress = {row['disease_name']: row.to_dict() for _, row in done_df.iterrows()}
else:
    progress = {}

print(f"Resuming: {len(progress)}/{len(df)} done")

for idx, row in df.iterrows():
    name = str(row['disease_name']).strip()
    if name in progress:
        continue

    result = clean_row(name, row.get('symptoms'), row.get('risk_factors'))
    progress[name] = {
        'disease_name': name,
        'symptoms': result.get('symptoms', ''),
        'risk_factors': result.get('risk_factors', ''),
        'specialist': result.get('specialist', ''),
    }

    if len(progress) % 10 == 0:
        pd.DataFrame(progress.values()).to_csv(PROGRESS_FILE, index=False)
        print(f"  {len(progress)}/{len(df)} ({100*len(progress)/len(df):.0f}%)")

    time.sleep(0.3)

pd.DataFrame(progress.values()).to_csv(PROGRESS_FILE, index=False)
print(f"Done: {len(progress)}/{len(df)}")

Resuming: 850/1177 done
  860/1177 (73%)
  870/1177 (74%)
  880/1177 (75%)
  890/1177 (76%)
  900/1177 (76%)
  910/1177 (77%)
  920/1177 (78%)
  930/1177 (79%)
  940/1177 (80%)
  950/1177 (81%)
  960/1177 (82%)
  970/1177 (82%)
  980/1177 (83%)
  990/1177 (84%)
  1000/1177 (85%)
  1010/1177 (86%)
  1020/1177 (87%)
  1030/1177 (88%)
  1040/1177 (88%)
  1050/1177 (89%)
  1060/1177 (90%)
  1070/1177 (91%)
  1080/1177 (92%)
  1090/1177 (93%)
  1100/1177 (93%)
  1110/1177 (94%)
  1120/1177 (95%)
  1130/1177 (96%)
  1140/1177 (97%)
  1150/1177 (98%)
  1160/1177 (99%)
  1170/1177 (99%)
Done: 1177/1177


In [10]:
# Cell 4: Build final CSV

clean_df = pd.DataFrame(progress.values())[['disease_name', 'symptoms', 'risk_factors', 'specialist']]

print(f"Shape: {clean_df.shape}")
print(f"Empty symptoms: {(clean_df['symptoms'] == '').sum()}")
print(f"Empty risk_factors: {(clean_df['risk_factors'] == '').sum()}")
print(f"\nSpecialist distribution:\n{clean_df['specialist'].value_counts()}")

OUTPUT = f'{BASE_DIR}/mayo_clinic_diseases_cleaned.csv'
clean_df.to_csv(OUTPUT, index=False)
print(f"\nSaved to {OUTPUT}")

Shape: (1177, 4)
Empty symptoms: 3
Empty risk_factors: 26

Specialist distribution:
specialist
neurologist                       112
oncologist                        110
gastroenterologist                 98
dermatologist                      92
cardiologist                       85
orthopedic surgeon                 67
psychiatrist                       58
urologist                          52
endocrinologist                    47
neurosurgeon                       41
pulmonologist                      40
gynecologist                       38
rheumatologist                     37
ophthalmologist                    35
infectious disease specialist      29
otolaryngologist                   23
nephrologist                       21
vascular surgeon                   20
allergist                          18
sleep medicine specialist          15
hepatologist                       13
hematologist                       12
obstetrician                       12
dentist                        